# Library Imports

In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import image as mpimg

# Useful Functions

In [2]:

# Fourier Transformation Utility Functions using NumPy
def fft2(u):
    return np.fft.fftshift(np.fft.fft2(u))

def ifft2(u_hat):
    return np.fft.ifft2(np.fft.ifftshift(u_hat))


# Central Fractional Derivatives Implementation as described in the report

def Dx_central(u_hat, alpha, M, N):
    size = M
    middle_row_index = size//2
    matrix = np.zeros((size, size), dtype=int)

    for i in range(middle_row_index):
        if (size%2) :
            matrix[i] = - size + i +1 + middle_row_index
        else :
            matrix[i] = - size + i + middle_row_index
    for i in range(middle_row_index + 1, size):
        matrix[i] = + i - middle_row_index 

    x = np.power(1 - np.exp(-1j*2*np.pi * matrix/M), alpha)
    z = np.power(np.exp(1j*np.pi * matrix/M), alpha)
    dx_hat = u_hat * x * z
    dx = ifft2(dx_hat)
    return dx

def Dy_central(u_hat, alpha, M, N):
    size = N
    middle_row_index = size//2
    matrix = np.zeros((size, size), dtype=int)

    for i in range(middle_row_index):
        if (size%2) :
            matrix[:,i] = - size + i +1 + middle_row_index
        else :
            matrix[:,i] = - size + i + middle_row_index
    for i in range(middle_row_index + 1, size):
        matrix[:,i] = + i - middle_row_index 

    y = np.power(1 - np.exp(-1j*2*np.pi * matrix/N), alpha)
    z = np.power(np.exp(1j*np.pi * matrix/N), alpha)
    dy_hat = u_hat * y * z
    dy = ifft2(dy_hat)
    return dy

# Compute Kx and Ky as the described operators in the report.
def make_K(M, N, alpha):
    w1,w2 = np.mgrid[0:M, 0:N]
    w1_grid = w1 - (M/2)*np.ones((M,N))
    w2_grid = w2 - (N/2)*np.ones((M,N))
    Kx = np.power(1 - np.exp(-1j*2*np.pi * w1_grid/M), alpha)*np.power(np.exp(1j*np.pi * w1_grid/M), alpha)
    Ky = np.power(1 - np.exp(-1j*2*np.pi * w2_grid/N), alpha)*np.power(np.exp(1j*np.pi * w2_grid/N), alpha)
    return Kx, Ky

# Save the output image in grayscale format
def saveImg(img, file_path):

    # Check if img is a 2D grayscale or a 3D RGB where all channels are identical
    if len(img.shape) == 3 and np.all(img[:, :, 0] == img[:, :, 1]) and np.all(img[:, :, 1] == img[:, :, 2]):
        # If 3D RGB with identical channels, take the first channel to convert to grayscale
        img = img[:, :, 0]

    # Normalize the image to 0-1 range if it's not already in uint8 format
    if img.dtype != np.uint8:
        img = (img - np.min(img)) / (np.max(img) - np.min(img))

    # Save the image
    mpimg.imsave(file_path, img, cmap='gray')

# Parameters

In [3]:
dt = 0.01
alpha = 2.2

# Load and Process the Image

In [ ]:
# Load the clean image and add noise to it
u_clean = cv2.imread('barbara.png', cv2.IMREAD_GRAYSCALE)
std = (7 * np.max(u_clean))/100
noise = np.random.normal(0, std, u_clean.shape)
u_noisy = u_clean + noise

# Execution

In [ ]:
# Set the noisy image as the starting point
u = u_noisy.copy()
M, N = u.shape
Kx, Ky = make_K(M, N, alpha)

# Compute the initial PSNR
prev_PSNR = cv2.PSNR(u_clean.astype('float32'), u.astype('float32'))

i = 0 # variable to keep track of th iterations

# Compute u_hat
u_hat = fft2(u)

while True:
    i += 1
    dx = Dx_central(u_hat, alpha, M, N)
    dy = Dy_central(u_hat, alpha, M, N)
    D = np.square(dx.real) + np.square(dy.real)
    c = 1/(1+D)
    gx = c * dx
    gy = c * dy
    gx_hat = fft2(gx)
    gy_hat = fft2(gy)
    g_hat = (np.conj(Kx) * gx_hat) + (np.conj(Ky) * gy_hat)
    u_hat = u_hat - g_hat * dt
    u = ifft2(u_hat).real

    # Compute the current PSNR
    current_PSNR = cv2.PSNR(u_clean.astype('float32'), u.astype('float32'))
    print(f"Iteration {i}", current_PSNR - prev_PSNR)
    
    # Saving the intermediate images too for better understanding
    if(i%4000 == 0):
        version = f"{int(i/4000)}"
        alpha_str = f"{alpha}".replace('.','_')[:3]
        file_path = f"{alpha_str}_version_{version}.png"
        saveImg(u,file_path)
    
    # Check for the stopping criteria
    if (current_PSNR - prev_PSNR) < 1e-5:
        alpha_str = f"{alpha}".replace('.','_')[:3]
        file_path = f"{alpha_str}_final.png"
        saveImg(u,file_path)
        break
    else:
        prev_PSNR = current_PSNR

# Plot the images

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

# Plot the original image
ax1.imshow(u_clean, cmap='gray')
ax1.set_title('Clean Image')

# Plot the denoised image
ax2.imshow(u_denoised, cmap='gray')
ax2.set_title('Denoised Image')

# Plot the noisy image
ax3.imshow(u_noisy, cmap='gray')
ax3.set_title('Noisy Image')

plt.show()